In [0]:
# =====================================================
# Imports
# =====================================================

from delta.tables import DeltaTable

from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col,
    from_json,
    size,
    when,
    lit,
    current_timestamp,
    posexplode,
    arrays_zip,
    explode,
    to_timestamp,
    sha2,
    concat_ws,
    coalesce,
    row_number,
    desc
)

from pyspark.sql.types import (
    StructType,
    StructField,
    ArrayType,
    StringType,
    DoubleType,
    LongType,
    IntegerType
)

In [0]:
# =====================================================
# Constants
# =====================================================

BRONZE_TABLE_NAME = "weather_project.north_texas_weather.bronze_weather_raw"

SILVER_TABLE_NAME = "weather_project.north_texas_weather.silver_hourly_weather"

QUARANTINE_TABLE_NAME = "weather_project.north_texas_weather.silver_weather_rejections"

MIN_TEMPERATURE_C = -100.0
MAX_TEMPERATURE_C = 100.0

MIN_PRECIPITATION_MM = 0.0
MAX_PRECIPITATION_MM = 500.0

In [0]:
# =====================================================
# Configure Spark
# =====================================================

spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
# =====================================================
# Runtime parameters
# =====================================================

dbutils.widgets.text("ingestion_id", "") # Change 2nd param to "ALL" to read the full bronze table (to test/rebuild silver)
dbutils.widgets.text("has_new_data", "true")

INGESTION_ID = dbutils.widgets.get("ingestion_id").strip()

HAS_NEW_DATA = dbutils.widgets.get("has_new_data").strip().lower()

if HAS_NEW_DATA == "false":
    dbutils.notebook.exit("NO_NEW_BRONZE_INGESTION")

if not INGESTION_ID:
    raise RuntimeError("Silver requires an ingestion_id from the upstream Bronze task")

if INGESTION_ID.upper() == "ALL":
    bronze_df = spark.read.table(BRONZE_TABLE_NAME)
else:
    bronze_df = (
        spark.read
        .table(BRONZE_TABLE_NAME)
        .filter(col("ingestion_id") == INGESTION_ID)
    )

if bronze_df.limit(1).count() == 0:
    raise RuntimeError(f"No Bronze ingestion found for ingestion_id={INGESTION_ID}")

In [0]:
# =====================================================
# 1. Schemas
# =====================================================

hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),
    StructField("temperature_2m", ArrayType(DoubleType()), True),
    StructField("precipitation", ArrayType(DoubleType()), True)
])

hourly_units_schema = StructType([
    StructField("time", StringType(), True),
    StructField("temperature_2m", StringType(), True),
    StructField("precipitation", StringType(), True)
])

location_response_schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("elevation", DoubleType(), True),
    StructField("generationtime_ms", DoubleType(), True),
    StructField("utc_offset_seconds", LongType(), True),
    StructField("timezone", StringType(), True),
    StructField("timezone_abbreviation", StringType(), True),
    StructField("hourly", hourly_schema, True),
    StructField("hourly_units", hourly_units_schema, True)
])

open_meteo_response_schema = ArrayType(location_response_schema)

request_location_schema = StructType([
    StructField("location_id", StringType(), False),
    StructField("city", StringType(), False),
    StructField("state_code", StringType(), False),
    StructField("country_code", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("request_order", IntegerType(), False)
])

request_locations_schema = ArrayType(request_location_schema)

# =====================================================
# 2. Parse Bronze JSON Strings
# =====================================================

parsed_bronze_df = (
    bronze_df
    .withColumn(
        "source_locations",
        from_json(
            col("raw_response"),
            open_meteo_response_schema
        )
    )
    .withColumn(
        "requested_locations",
        from_json(
            col("request_locations"),
            request_locations_schema
        )
    )
)

display(parsed_bronze_df)

In [0]:
# =====================================================
# 1. Validate ingestion-level structure
# =====================================================

ingestion_checked_df = (
    parsed_bronze_df
    .withColumn(
        "ingestion_dq_reason",
        when(
            col("source_locations").isNull(),
            lit("MALFORMED_RAW_RESPONSE")
        )
        .when(
            col("requested_locations").isNull(),
            lit("MALFORMED_REQUEST_LOCATIONS")
        )
        .when(
            size(col("source_locations")) == 0,
            lit("EMPTY_SOURCE_RESPONSE")
        )
        .when(
            size(col("source_locations")) != size(col("requested_locations")),
            lit("LOCATION_COUNT_MISMATCH")
        )
        .otherwise(lit(None))
    )
)

# =====================================================
# 2. Separate valid and invalid ingestions
# =====================================================

valid_ingestions_df = (
    ingestion_checked_df
    .filter(
        col("ingestion_dq_reason").isNull()
    )
)

invalid_ingestions_df = (
    ingestion_checked_df
    .filter(
        col("ingestion_dq_reason").isNotNull()
    )
)

display(invalid_ingestions_df)

In [0]:
# =====================================================
# 1. Explode open-meteo locations with array position
# =====================================================

source_locations_df = (
    valid_ingestions_df
    .select(
        col("ingestion_id"),
        col("ingested_at"),
        col("source"),
        col("model"),
        col("request_start_date"),
        col("request_end_date"),

        posexplode(
            col("source_locations")
        ).alias(
            "location_index",
            "source_location"
        )
    )
)

# =====================================================
# 2. Explode requested locations with array position
# =====================================================

requested_locations_df = (
    valid_ingestions_df
    .select(
        col("ingestion_id"),

        posexplode(
            col("requested_locations")
        ).alias(
            "location_index",
            "requested_location"
        )
    )
)

# =====================================================
# 3. Match each API response to its requested location
# =====================================================

location_df = (
    source_locations_df
    .join(
        requested_locations_df,
        on=[
            "ingestion_id",
            "location_index"
        ],
        how="inner"
    )
)

# =====================================================
# 4. Validate each location response
# =====================================================

location_checked_df = (
    location_df
    .withColumn(
        "location_dq_reason",
        when(
            col("requested_location.location_id").isNull(),
            lit("MISSING_LOCATION_ID")
        )
        .when(
            col("requested_location.city").isNull(),
            lit("MISSING_CITY")
        )
        .when(
            col("source_location.hourly").isNull(),
            lit("MISSING_HOURLY_OBJECT")
        )
        .when(
            size(col("source_location.hourly.time")) == 0,
            lit("EMPTY_HOURLY_ARRAY")
        )
        .when(
            col("source_location.hourly.time").isNull(),
            lit("MISSING_TIME_ARRAY")
        )
        .when(
            col("source_location.hourly.temperature_2m").isNull(),
            lit("MISSING_TEMPERATURE_ARRAY")
        )
        .when(
            col("source_location.hourly.precipitation").isNull(),
            lit("MISSING_PRECIPITATION_ARRAY")
        )
        .when(
            size(col("source_location.hourly.time")) != size(col("source_location.hourly.temperature_2m")),
            lit("HOURLY_ARRAY_LENGTH_MISMATCH")
        )
        .when(
            size(col("source_location.hourly.time")) != size(col("source_location.hourly.precipitation")),
            lit("HOURLY_ARRAY_LENGTH_MISMATCH")
        )
        .when(
            col("source_location.utc_offset_seconds").isNull(),
            lit("MISSING_UTC_OFFSET")
        )
        .when(
            col("source_location.utc_offset_seconds") != 0,
            lit("NON_UTC_RESPONSE")
        )
        .when(
            col("source_location.hourly_units.temperature_2m").isNull(),
            lit("MISSING_TEMPERATURE_UNITS")
        )
        .when(
            col("source_location.hourly_units.temperature_2m") != "°C",
            lit("UNEXPECTED_TEMPERATURE_UNIT")
        )
        .when(
            col("source_location.hourly_units.precipitation").isNull(),
            lit("MISSING_PRECIPITATION_UNITS")
        )
        .when(
            col("source_location.hourly_units.precipitation") != "mm",
            lit("UNEXPECTED_PRECIPITATION_UNIT")
        )
        .otherwise(lit(None))
    )
)

valid_locations_df = (
    location_checked_df
    .filter(
        col("location_dq_reason").isNull()
    )
)

invalid_locations_df = (
    location_checked_df
    .filter(
        col("location_dq_reason").isNotNull()
    )
)

In [0]:
# =====================================================
# 1. Zip corresponding hourly arrays
# =====================================================

hourly_zipped_df = (
    valid_locations_df
    .withColumn(
        "weather_records",
        arrays_zip(
            col("source_location.hourly.time"),
            col("source_location.hourly.temperature_2m"),
            col("source_location.hourly.precipitation")
        )
    )
)

# =====================================================
# 2. Explode into one row per hourly observation
# =====================================================

hourly_exploded_df = (
    hourly_zipped_df
    .select(
        col("ingestion_id"),
        col("ingested_at"),
        col("source"),
        col("model"),
        col("location_index"),
        col("request_start_date"),
        col("request_end_date"),
        col("requested_location.location_id").alias("location_id"),
        col("requested_location.city").alias("city"),
        col("requested_location.state_code").alias("state_code"),
        col("requested_location.country_code").alias("country_code"),
        col("requested_location.latitude").alias("requested_latitude"),
        col("requested_location.longitude").alias("requested_longitude"),
        col("source_location.latitude").alias("source_latitude"),
        col("source_location.longitude").alias("source_longitude"),
        col("source_location.elevation").alias("source_elevation"),
        explode(col("weather_records")).alias("weather_record")
    )
)

silver_candidate_df = (
    hourly_exploded_df
    .select(
        col("location_id"),
        col("city"),
        col("state_code"),
        col("country_code"),
        col("source"),
        col("location_index"),
        col("request_start_date"),
        col("request_end_date"),
        col("requested_latitude"),
        col("requested_longitude"),
        col("source_latitude"),
        col("source_longitude"),
        col("source_elevation"),
        col("weather_record.time").alias("timestamp_raw"),
        col("weather_record.temperature_2m").alias("temperature_celsius"),
        col("weather_record.precipitation").alias("precipitation_mm"),
        col("model"),
        col("ingestion_id"),
        col("ingested_at")
    )
)

silver_candidate_df = (
    silver_candidate_df
    .withColumn(
        "timestamp_utc",
        to_timestamp(col("timestamp_raw"), "yyyy-MM-dd'T'HH:mm")
    )
    .withColumn(
        "temperature_fahrenheit",
        (col("temperature_celsius") * 9.0 / 5.0) + 32.0
    )
    .withColumn(
        "precipitation_inches",
        col("precipitation_mm") / 25.4
    )
)

# =====================================================
# 3. Data Quality guardrails
# =====================================================

silver_checked_df = (
    silver_candidate_df
    .withColumn(
        "observation_dq_reason",
        when(
            col("timestamp_utc").isNull(),
            lit("INVALID_TIMESTAMP")
        )
        .when(
            col("temperature_celsius").isNull(),
            lit("MISSING_TEMPERATURE")
        )
        .when(
            col("precipitation_mm").isNull(),
            lit("MISSING_PRECIPITATION")
        )
        .when(
           (col("temperature_celsius") < MIN_TEMPERATURE_C) | (col("temperature_celsius") > MAX_TEMPERATURE_C),
            lit("TEMPERATURE_OUT_OF_RANGE")
        )
        .when(
            (col("precipitation_mm") < MIN_PRECIPITATION_MM) | (col("precipitation_mm") > MAX_PRECIPITATION_MM),
            lit("PRECIPITATION_OUT_OF_RANGE")
        )
        .otherwise(lit(None))
    )
)

valid_observations_df = (
    silver_checked_df
    .filter(
        col("observation_dq_reason").isNull()
    )
)

invalid_observations_df = (
    silver_checked_df
    .filter(
        col("observation_dq_reason").isNotNull()
    )
)

In [0]:
# =================================================================
# Quarantine table for ingests, locations, and observations
# =================================================================

ingestion_rejections_df = (
    invalid_ingestions_df
    .select(
        col("ingestion_id"),
        col("ingested_at"),
        col("source"),
        col("model"),

        col("request_start_date"),
        col("request_end_date"),

        lit("INGESTION").alias(
            "rejection_level"
        ),

        col("ingestion_dq_reason").alias(
            "rejection_reason"
        ),

        lit(None)
        .cast("integer")
        .alias("location_index"),

        lit(None)
        .cast("string")
        .alias("location_id"),

        lit(None)
        .cast("string")
        .alias("city"),

        lit(None)
        .cast("string")
        .alias("timestamp_raw"),

        lit(None)
        .cast("double")
        .alias("temperature_celsius"),

        lit(None)
        .cast("double")
        .alias("precipitation_mm")
    )
)

location_rejections_df = (
    invalid_locations_df
    .select(
        col("ingestion_id"),
        col("ingested_at"),
        col("source"),
        col("model"),

        col("request_start_date"),
        col("request_end_date"),

        lit("LOCATION").alias(
            "rejection_level"
        ),

        col("location_dq_reason").alias(
            "rejection_reason"
        ),

        col("location_index"),

        col(
            "requested_location.location_id"
        ).alias("location_id"),

        col(
            "requested_location.city"
        ).alias("city"),

        lit(None)
        .cast("string")
        .alias("timestamp_raw"),

        lit(None)
        .cast("double")
        .alias("temperature_celsius"),

        lit(None)
        .cast("double")
        .alias("precipitation_mm")
    )
)

observation_rejections_df = (
    invalid_observations_df
    .select(
        col("ingestion_id"),
        col("ingested_at"),
        col("source"),
        col("model"),

        col("request_start_date"),
        col("request_end_date"),

        lit("OBSERVATION").alias(
            "rejection_level"
        ),

        col("observation_dq_reason").alias(
            "rejection_reason"
        ),

        col("location_index"),
        col("location_id"),
        col("city"),

        col("timestamp_raw"),

        col("temperature_celsius"),
        col("precipitation_mm")
    )
)

quarantine_df = (
    ingestion_rejections_df

    .unionByName(
        location_rejections_df
    )

    .unionByName(
        observation_rejections_df
    )
)

quarantine_df = (
    quarantine_df

    .withColumn(
        "rejection_id",

        sha2(
            concat_ws(
                "||",

                col("ingestion_id"),

                col("rejection_level"),

                col("rejection_reason"),

                coalesce(
                    col("location_index")
                    .cast("string"),
                    lit("")
                ),

                coalesce(
                    col("location_id"),
                    lit("")
                ),

                coalesce(
                    col("timestamp_raw"),
                    lit("")
                )
            ),
            256
        )
    )

    .withColumn(
        "rejected_at",
        current_timestamp()
    )
)

spark.sql("""
CREATE TABLE IF NOT EXISTS
weather_project.north_texas_weather.silver_weather_rejections
(
    rejection_id STRING NOT NULL,

    ingestion_id STRING NOT NULL,
    ingested_at TIMESTAMP NOT NULL,

    source STRING,
    model STRING,

    request_start_date DATE,
    request_end_date DATE,

    rejection_level STRING NOT NULL,
    rejection_reason STRING NOT NULL,

    location_index INT,
    location_id STRING,
    city STRING,

    timestamp_raw STRING,

    temperature_celsius DOUBLE,
    precipitation_mm DOUBLE,

    rejected_at TIMESTAMP NOT NULL
)
USING DELTA
""")

quarantine_target = DeltaTable.forName(
    spark,
    QUARANTINE_TABLE_NAME
)

if quarantine_df.limit(1).count() > 0:

    (
        quarantine_target
        .alias("target")

        .merge(
            quarantine_df.alias("source"),

            """
            target.rejection_id =
            source.rejection_id
            """
        )

        .whenNotMatchedInsertAll()

        .execute()
    )

In [0]:
# =====================================================
# 1. Define observation natural key
# =====================================================

deduplication_window = (
    Window
    .partitionBy(
        "location_id",
        "timestamp_utc"
    )
    .orderBy(
        desc("ingested_at"),
        desc("ingestion_id")
    )
)

# =====================================================
# 2. Keep newest ingestion for each observation
# =====================================================

silver_clean_df = (
    valid_observations_df
    .withColumn(
        "_row_number",
        row_number().over(deduplication_window)
    )
    .filter(col("_row_number") == 1)
    .drop(
        "_row_number",
        "timestamp_raw",
        "observation_dq_reason"
    )
)

In [0]:
# ==================================================================
# Explicitly create first table instead of inferring on first write
# ==================================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS
weather_project.north_texas_weather.silver_hourly_weather
(
    location_id STRING NOT NULL,

    city STRING NOT NULL,
    state_code STRING NOT NULL,
    country_code STRING NOT NULL,

    requested_latitude DOUBLE NOT NULL,
    requested_longitude DOUBLE NOT NULL,

    source_latitude DOUBLE,
    source_longitude DOUBLE,
    source_elevation DOUBLE,

    timestamp_utc TIMESTAMP NOT NULL,

    temperature_celsius DOUBLE NOT NULL,
    temperature_fahrenheit DOUBLE NOT NULL,

    precipitation_mm DOUBLE NOT NULL,
    precipitation_inches DOUBLE NOT NULL,

    model STRING NOT NULL,

    ingestion_id STRING NOT NULL,
    ingested_at TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
# ==================================================================
# Write final silver table
# ==================================================================

silver_target = DeltaTable.forName(spark, SILVER_TABLE_NAME)

silver_final_df = (
    silver_clean_df
    .select(
        col("location_id"),

        col("city"),
        col("state_code"),
        col("country_code"),

        col("requested_latitude"),
        col("requested_longitude"),

        col("source_latitude"),
        col("source_longitude"),
        col("source_elevation"),

        col("timestamp_utc"),

        col("temperature_celsius"),
        col("temperature_fahrenheit"),

        col("precipitation_mm"),
        col("precipitation_inches"),

        col("model"),

        col("ingestion_id"),
        col("ingested_at")
    )
)

(
    silver_target
    .alias("target")

    .merge(
        silver_final_df.alias("source"),

        """
        target.location_id =
        source.location_id

        AND target.timestamp_utc =
        source.timestamp_utc
        """
    )

    .whenMatchedUpdateAll(
        condition="""
        source.ingested_at >=
        target.ingested_at
        """
    )

    .whenNotMatchedInsertAll()

    .execute()
)